# Model Training Pipeline for basic models

In [1]:
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import sys
import json

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS, GENERAL_KEYS, AABB_KEYS, TFBB_KEYS, TOPO_KEYS, MATERIAL_KEYS, RAY_KEYS
from models_helper import ModelTrainer, FeatureGroupEvaluator, collect_model_metrics
from dataviz_helper import plot_confusion_matrices, plot_feature_importance

In [2]:
DATA_DIR = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed.parquet")
df_train_over = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed_oversampled.parquet")
df_val   = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


In [3]:
FEATURE_GROUPS = {
    "geom": GENERAL_KEYS,
    "aabb": AABB_KEYS,
    "tfbb": TFBB_KEYS,
    "topo": TOPO_KEYS,
    "material": MATERIAL_KEYS,
    "ray": RAY_KEYS
}

BASELINE_MODELS = [
    ("naive_bayes", "Naive Bayes"),
    ("logistic_regression", "Logistic Regression"),
    ("knn", "KNN"),
    ("svm", "SVM"),
    ("decision_tree", "Decision Tree"),
    ("random_forest", "Random Forest"),
    ("extra_trees", "Extra Trees"),
]

In [4]:
label_cols = [c for c in df_train.columns if c.startswith("label_")]

X_train = df_train[ALL_FEATURE_KEYS]
y_train = df_train[label_cols]
X_train_over = df_train_over[ALL_FEATURE_KEYS]
y_train_over = df_train_over[label_cols]
X_val   = df_val[ALL_FEATURE_KEYS]
y_val   = df_val[label_cols]

print(f"Train: {len(X_train)}, Train (oversampled): {len(X_train_over)}, Val: {len(X_val)}")
print(f"Labels: {label_cols}")

Train: 26977, Train (oversampled): 28756, Val: 8458
Labels: ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 1. Train Models

### 1.1 Naive Bayes

In [5]:
trainer_nb = ModelTrainer("naive_bayes", label_cols)
trainer_nb.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.3682,0.2699,0.2145,0.1846,0.3135,0.2365,0.1875,0.1690
label_predefined_type,0.3866,0.2237,0.1752,0.1493,0.3692,0.2156,0.1620,0.1438
label_is_external,0.1723,0.3150,0.1814,0.3304,0.0987,0.2548,0.1452,0.2983
label_load_bearing,0.0481,0.1100,0.0032,0.1887,0.0439,0.1114,0.0022,0.1884
mean,0.2438,0.2296,0.1436,0.2132,0.2063,0.2046,0.1242,0.1999


### 1.2 Logistic Regression

In [6]:
trainer_lr = ModelTrainer("logistic_regression", label_cols)
trainer_lr.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9461,0.8872,0.7398,0.5928,0.9435,0.9148,0.7271,0.5956
label_predefined_type,0.7950,0.7277,0.4388,0.4460,0.7959,0.8121,0.4376,0.4634
label_is_external,0.7989,0.8901,0.6374,0.7899,0.7817,0.8765,0.6398,0.8010
label_load_bearing,0.7989,0.8470,0.6382,0.7361,0.7648,0.8206,0.6210,0.7249
mean,0.8347,0.8380,0.6136,0.6412,0.8215,0.8560,0.6064,0.6462


### 1.3 KNN

In [7]:
trainer_knn = ModelTrainer("knn", label_cols)
trainer_knn.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9772,0.8782,0.6989,0.5530,0.9793,0.9635,0.7037,0.5898
label_predefined_type,0.9179,0.8717,0.4220,0.4050,0.9243,0.9157,0.4509,0.4566
label_is_external,0.9485,0.9728,0.6001,0.7785,0.9515,0.9744,0.6158,0.7804
label_load_bearing,0.9505,0.9582,0.6762,0.7908,0.9536,0.9615,0.6609,0.7780
mean,0.9485,0.9202,0.5993,0.6318,0.9522,0.9538,0.6078,0.6512


### 1.4 SVM

In [8]:
trainer_svm = ModelTrainer("svm", label_cols)
trainer_svm.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9719,0.8723,0.7239,0.5970,0.9724,0.9589,0.7280,0.6193
label_predefined_type,0.8674,0.7561,0.5125,0.4492,0.8707,0.8499,0.5690,0.5649
label_is_external,0.9217,0.9556,0.7521,0.8702,0.9220,0.9557,0.7364,0.8587
label_load_bearing,0.9178,0.9299,0.7356,0.8248,0.9139,0.9298,0.7197,0.8172
mean,0.9197,0.8785,0.6810,0.6853,0.9198,0.9236,0.6883,0.7150


### 1.5 Decision Tree

In [9]:
trainer_dt = ModelTrainer("decision_tree", label_cols)
trainer_dt.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9997,0.9996,0.6780,0.4762,0.9998,0.9997,0.7107,0.5342
label_predefined_type,0.9991,0.9993,0.4451,0.4508,0.9992,0.9994,0.4218,0.4101
label_is_external,0.9999,0.9999,0.5661,0.7705,0.9999,0.9999,0.6177,0.8036
label_load_bearing,0.9990,0.9992,0.6078,0.7427,0.9990,0.9992,0.6616,0.7749
mean,0.9994,0.9995,0.5742,0.6100,0.9995,0.9996,0.6030,0.6307


### 1.6 Random Forest

In [10]:
trainer_rf = ModelTrainer("random_forest", label_cols)
trainer_rf.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9996,0.9993,0.7370,0.6336,0.9996,0.9996,0.7581,0.6446
label_predefined_type,0.9990,0.9989,0.4773,0.4846,0.9990,0.9992,0.4790,0.4966
label_is_external,0.9999,0.9999,0.7250,0.8565,0.9999,0.9999,0.7357,0.8620
label_load_bearing,0.9988,0.9991,0.7394,0.8237,0.9988,0.9991,0.7373,0.8215
mean,0.9993,0.9993,0.6697,0.6996,0.9993,0.9994,0.6775,0.7062


### 1.7 Extra Trees

In [11]:
trainer_et = ModelTrainer("extra_trees", label_cols)
trainer_et.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9997,0.9996,0.7847,0.6217,0.9998,0.9997,0.7886,0.6247
label_predefined_type,0.9991,0.9993,0.4841,0.4727,0.9992,0.9994,0.4817,0.4661
label_is_external,0.9999,0.9999,0.7343,0.8616,0.9999,0.9999,0.7489,0.8699
label_load_bearing,0.9990,0.9992,0.7833,0.8608,0.9990,0.9992,0.7801,0.8574
mean,0.9994,0.9995,0.6966,0.7042,0.9995,0.9996,0.6998,0.7045


## 2. Analyse Confusion Matrix ond best model (Random Forest)

In [12]:
# plot confusion matrices for random forest ond oversampled dataset
trainer_rf = ModelTrainer("random_forest", label_cols)
trainer_rf.fit(X_train_over, y_train_over)
cms = trainer_rf.confusion_matrices(X_val, y_val)
plot_confusion_matrices(cms, label="ifc_entity", ncols=1)

In [13]:
plot_confusion_matrices(cms, label="predefined_type", ncols=1)

In [14]:
plot_confusion_matrices(cms, label=["is_external", "load_bearing"], ncols=2, height=400)

## 3. Feature Importance from best model (Random Forest)

In [15]:
# plot feature importance for random forest ond oversampled dataset
df_imp = trainer_rf.feature_importances(ALL_FEATURE_KEYS)
plot_feature_importance(df_imp, FEATURE_GROUPS)

In [16]:
# save feature importance to json
df_imp.to_json("feature_importance_random_forest.json", orient="records", indent=2)

## 4. Try different Feature Group Combinations with best model (Random Forest)

In [17]:
# random forest as default model for feature group evaluation on oversampled dataset
evaluator = FeatureGroupEvaluator(FEATURE_GROUPS, label_cols)
results = evaluator.evaluate(X_train_over, y_train_over, X_val, y_val)

# save feature group evaluation results to json (index keeps the combination names)
results.reset_index().to_json("feature_group_evaluation_random_forest.json", orient="records", indent=2)

In [18]:
results.head(n=30)

,n_features,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,mean_f1_macro
features,,,,,,
geom+topo+material+ray,46,0.636,0.488,0.859,0.852,0.709
geom+aabb+topo+material+ray,60,0.644,0.496,0.867,0.820,0.707
geom+material+ray,37,0.636,0.477,0.864,0.850,0.707
geom+aabb+tfbb+topo+material+ray,73,0.650,0.487,0.858,0.824,0.705
aabb+topo+material,46,0.628,0.507,0.865,0.807,0.702
geom+aabb+material+ray,51,0.634,0.469,0.878,0.820,0.700
aabb+topo+material+ray,48,0.636,0.507,0.848,0.809,0.700
aabb+tfbb+topo+material+ray,61,0.636,0.492,0.862,0.805,0.699
geom+aabb+topo+material,58,0.647,0.470,0.853,0.819,0.697


## 5. Collect metrics of all baseline models for the report

In [19]:
# collect metrics for all baseline models on oversampled dataset
baseline_metrics = []
for key, display in BASELINE_MODELS:
    print(f"collecting metrics for {key}...")
    baseline_metrics.append(collect_model_metrics(key, label_cols, X_train_over, y_train_over, X_val, y_val, display_name=display))

with open("baseline_model_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

pd.DataFrame(baseline_metrics).set_index("model")

collecting metrics for naive_bayes...
collecting metrics for logistic_regression...
collecting metrics for knn...
collecting metrics for svm...
collecting metrics for decision_tree...
collecting metrics for random_forest...
collecting metrics for extra_trees...


,train_mcc,train_f1_macro,val_mcc,val_f1_macro,val_precision,val_accuracy
model,,,,,,
Naive Bayes,0.2063,0.2046,0.1242,0.1999,0.3086,0.2966
Logistic Regression,0.8215,0.8560,0.6064,0.6462,0.7768,0.6992
KNN,0.9522,0.9538,0.6078,0.6512,0.7720,0.7060
SVM,0.9198,0.9236,0.6883,0.7150,0.8232,0.7689
Decision Tree,0.9995,0.9996,0.6030,0.6307,0.7464,0.7031
Random Forest,0.9993,0.9994,0.6775,0.7062,0.8137,0.7507
Extra Trees,0.9995,0.9996,0.6998,0.7045,0.8259,0.7710
